# 🏡 House Prices Prediction with XGBoost  
This notebook solves the **House Prices - Advanced Regression Techniques** competition on Kaggle.  
We predict house sale prices using **feature engineering, data preprocessing, and XGBoost regression**.  

The workflow includes:  
1. Importing libraries  
2. Loading the data  
3. Handling missing values  
4. Feature engineering  
5. Encoding categorical variables  
6. Fixing skewness in numeric features  
7. Training an XGBoost model  
8. Cross-validation for model evaluation  
9. Final prediction and submission  

---



In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import skew


## 📂 Step 2: Load the Data  
We load the training, testing, and sample submission files provided by Kaggle.  


In [ ]:
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
sample_submission = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")


## 🔧 Step 3: Handle Missing Values  
- Fill missing categorical values with `"None"`  
- Fill missing numerical values with **median**  


In [ ]:
full_data = pd.concat([train.drop("SalePrice", axis=1), test], axis=0)

for col in full_data.columns:
    if full_data[col].dtype == "object":
        full_data[col] = full_data[col].fillna("None")
    else:
        full_data[col] = full_data[col].fillna(full_data[col].median())


## 🏗️ Step 4: Feature Engineering  
We create additional features to better represent the data:  
- `TotalSF`: total square footage  
- `HouseAge`: age of the house  
- `RemodAge`: years since remodel  
- `TotalBath`: total bathrooms (including basement)  
- `Qual*SF`: interaction of quality × living area  


In [ ]:
full_data["TotalSF"] = full_data["TotalBsmtSF"] + full_data["1stFlrSF"] + full_data["2ndFlrSF"]
full_data["HouseAge"] = full_data["YrSold"] - full_data["YearBuilt"]
full_data["RemodAge"] = full_data["YrSold"] - full_data["YearRemodAdd"]
full_data["TotalBath"] = (full_data["FullBath"] + 0.5*full_data["HalfBath"] +
                          full_data["BsmtFullBath"] + 0.5*full_data["BsmtHalfBath"])
full_data["Qual*SF"] = full_data["OverallQual"] * full_data["GrLivArea"]


## 🔑 Step 5: Encode Categorical Variables  
We use **Label Encoding** to convert categorical variables into numeric form.  


In [ ]:
for col in full_data.columns:
    if full_data[col].dtype == "object":
        lbl = LabelEncoder()
        full_data[col] = lbl.fit_transform(full_data[col].astype(str))


## 📉 Step 6: Fix Skewed Features  
We apply a **log1p transformation** on highly skewed numeric features to normalize them.  


In [ ]:
numeric_feats = full_data.dtypes[full_data.dtypes != "object"].index
skewed_feats = full_data[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewed = skewed_feats[abs(skewed_feats) > 0.75].index
full_data[skewed] = np.log1p(full_data[skewed])


## ✂️ Step 7: Train/Test Split  
- `X`: features for training  
- `y`: log-transformed target (SalePrice)  
- `X_test`: features for submission  


In [ ]:
X = full_data.iloc[:train.shape[0], :]
X_test = full_data.iloc[train.shape[0]:, :]
y = train["SalePrice"]

y_log = np.log1p(y)  # log-transform target


## ⚡ Step 8: Define XGBoost Model  
We use **XGBRegressor** with tuned hyperparameters for strong performance.  


In [ ]:
xgb_model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    reg_alpha=0.1,
    reg_lambda=0.1
)


## 📊 Step 9: Cross-Validation  
Evaluate performance using **10-fold cross-validation RMSE**.  


In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = -cross_val_score(xgb_model, X, y_log, scoring="neg_root_mean_squared_error", cv=kf)
print("Cross-validation RMSE:", cv_scores.mean())


## 🤖 Step 10–11: Train Model & Predict  
- Train the model on full training data  
- Predict log values → inverse log-transform back to SalePrice  


In [ ]:
xgb_model.fit(X, y_log)
preds = xgb_model.predict(X_test)
preds_final = np.expm1(preds)


## 📤 Step 12: Save Submission  
Final predictions are saved in `submission.csv` ready for Kaggle submission.  


In [ ]:
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds_final})
submission.to_csv("submission.csv", index=False)
print("Submission file created!")


In [ ]:
import joblib

# Save the trained model
joblib.dump(xgb_model, "house_price_xgb.pkl")
print("Model saved as house_price_xgb.pkl")
